# Function 3: Part 2 analysis notebook

This notebook keeps the original data-loading cells, appends the latest query point/output from the end of your uploaded notebook, and then runs one focused analysis for Part 2.

Assumption: lower output is better, so the optimisation target is minimisation.


In [2]:
import numpy as np

input_data = np.load('../../data/initial_data/function_3/initial_inputs.npy')
print("Before:", input_data.shape)
new_point = np.array([
    [0.754364, 0.233177, 0.303208],
    [0.312229, 0.060777, 0.000904],
    [0.5, 0.5, 0.5],
    ])
input_data = np.vstack([input_data, new_point])
print("After:", input_data.shape)
print(input_data)


Before: (15, 3)
After: (18, 3)
[[1.71525207e-01 3.43916870e-01 2.48737201e-01]
 [2.42114461e-01 6.44074270e-01 2.72432809e-01]
 [5.34905720e-01 3.98500915e-01 1.73388729e-01]
 [4.92581415e-01 6.11593188e-01 3.40176386e-01]
 [1.34621666e-01 2.19917240e-01 4.58206220e-01]
 [3.45523271e-01 9.41359831e-01 2.69363479e-01]
 [1.51836632e-01 4.39990619e-01 9.90881867e-01]
 [6.45502835e-01 3.97142940e-01 9.19771338e-01]
 [7.46911945e-01 2.84196309e-01 2.26299855e-01]
 [1.70476994e-01 6.97032401e-01 1.49169434e-01]
 [2.20549337e-01 2.97825244e-01 3.43555344e-01]
 [6.66013659e-01 6.71985151e-01 2.46295297e-01]
 [4.68089497e-02 2.31360241e-01 7.70617592e-01]
 [6.00097282e-01 7.25135725e-01 6.60886415e-02]
 [9.65994849e-01 8.61119690e-01 5.66829131e-01]
 [7.54364000e-01 2.33177000e-01 3.03208000e-01]
 [3.12229000e-01 6.07770000e-02 9.04000000e-04]
 [5.00000000e-01 5.00000000e-01 5.00000000e-01]]


In [ ]:
output_data = np.load('../../data/initial_data/function_3/initial_outputs.npy')
print("Before:", output_data.shape)
new_output = np.array([
    -0.09200841551496666,
    -0.18083748652026374,
    -0.015979341188442648
    ])
output_data = np.append(output_data, new_output)
print("After:", output_data.shape)
print(output_data)


Before: (15,)
After: (17,)
[-0.1121222  -0.08796286 -0.11141465 -0.03483531 -0.04800758 -0.11062091
 -0.39892551 -0.11386851 -0.13146061 -0.09418956 -0.04694741 -0.10596504
 -0.11804826 -0.03637783 -0.05675837 -0.09200842 -0.18083749]


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Latest query/output extracted from the uploaded notebook.
# Edit these if you later receive a different portal output.
latest_query = np.array([[0.5, 0.5, 0.5]])
actual_output = -0.015979341188442648

# Append the latest query/output only if it is not already present.
if actual_output is not None:
    already_present = np.any(np.all(np.isclose(input_data, latest_query, atol=1e-12), axis=1))
    if not already_present:
        input_data = np.vstack([input_data, latest_query])
        output_data = np.append(output_data, actual_output)

function_id = 3
d = input_data.shape[1]
print(f"Function {function_id}, dimension d={d}")
print("Data shape:", input_data.shape, output_data.shape)
print("Current best observed y:", output_data.min())
print("Current best x:", input_data[np.argmin(output_data)])


Function 3, dimension d=3
Data shape: (18, 3) (18,)
Current best observed y: -0.3989255131463011
Current best x: [0.15183663 0.43999062 0.99088187]


In [4]:
# Basic table used in all interpretations
summary = pd.DataFrame(input_data, columns=[f"x{i+1}" for i in range(d)])
summary["y"] = output_data
summary["log_abs_y"] = np.log(np.abs(output_data) + 1e-300)
summary["rank_min"] = summary["y"].rank(method="first", ascending=True).astype(int)
summary = summary.sort_values("y")
display(summary)


,x1,x2,x3,y,log_abs_y,rank_min
6,0.151837,0.439991,0.990882,-0.398926,-0.918981,1
16,0.312229,0.060777,0.000904,-0.180837,-1.710157,2
8,0.746912,0.284196,0.226300,-0.131461,-2.029048,3
12,0.046809,0.231360,0.770618,-0.118048,-2.136662,4
7,0.645503,0.397143,0.919771,-0.113869,-2.172711,5
0,0.171525,0.343917,0.248737,-0.112122,-2.188166,6
2,0.534906,0.398501,0.173389,-0.111415,-2.194496,7
5,0.345523,0.941360,0.269363,-0.110621,-2.201646,8
11,0.666014,0.671985,0.246295,-0.105965,-2.244646,9
9,0.170477,0.697032,0.149169,-0.094190,-2.362446,10


## Neural-network surrogate + input gradients

In [5]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# Fit a small neural network to y. If the output scale is extreme, use log|y| instead.
# This cell automatically chooses log|y| if the ratio of magnitudes is very large or if values are near zero.
X = input_data.astype(np.float32)
y_raw = output_data.astype(np.float32)
use_log_abs = (np.nanmax(np.abs(y_raw)) / max(np.nanmin(np.abs(y_raw) + 1e-300), 1e-300) > 1e4)

y_target = np.log(np.abs(y_raw) + 1e-300).astype(np.float32) if use_log_abs else y_raw.copy()
target_name = "log_abs_y" if use_log_abs else "y"

x_scaler = StandardScaler()
y_scaler = StandardScaler()
X_scaled = x_scaler.fit_transform(X).astype(np.float32)
y_scaled = y_scaler.fit_transform(y_target.reshape(-1, 1)).astype(np.float32).ravel()

X_t = torch.tensor(X_scaled, dtype=torch.float32)
y_t = torch.tensor(y_scaled.reshape(-1, 1), dtype=torch.float32)

class SurrogateNN(nn.Module):
    def __init__(self, d):
        super().__init__()
        width = max(16, 4*d)
        self.net = nn.Sequential(
            nn.Linear(d, width), nn.Tanh(),
            nn.Linear(width, width), nn.Tanh(),
            nn.Linear(width, 1)
        )
    def forward(self, x):
        return self.net(x)

torch.manual_seed(0)
model = SurrogateNN(d)
opt = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(3000):
    opt.zero_grad()
    pred = model(X_t)
    loss = loss_fn(pred, y_t)
    loss.backward()
    opt.step()

with torch.no_grad():
    pred_scaled = model(X_t).numpy().ravel()
    pred_target = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()

print("Target modelled:", target_name)
print(f"In-sample MSE in target space: {mean_squared_error(y_target, pred_target):.6g}")
print(f"In-sample R² in target space: {r2_score(y_target, pred_target):.4f}")


Target modelled: y
In-sample MSE in target space: 7.81256e-12
In-sample R² in target space: 1.0000


In [6]:
# Compute gradients of the network prediction with respect to original input variables.
X_grad = torch.tensor(X_scaled, dtype=torch.float32, requires_grad=True)
pred = model(X_grad)

# Sum is used so autograd gives one gradient per input point.
pred.sum().backward()
grad_scaled = X_grad.grad.detach().numpy()

# Convert from scaled-input / scaled-output gradient to original input scale.
grad_target = grad_scaled * (y_scaler.scale_[0] / x_scaler.scale_)
grad_norm = np.linalg.norm(grad_target, axis=1)

grad_df = pd.DataFrame(input_data, columns=[f"x{i+1}" for i in range(d)])
grad_df["y"] = output_data
grad_df[target_name] = y_target
for j in range(d):
    grad_df[f"grad_x{j+1}"] = grad_target[:, j]
grad_df["grad_norm"] = grad_norm
grad_df["dominant_variable"] = [f"x{np.argmax(np.abs(row))+1}" for row in grad_target]

display(grad_df.sort_values("grad_norm", ascending=False))

avg_abs_grad = np.mean(np.abs(grad_target), axis=0)
print("Average absolute gradient:")
for j, val in enumerate(avg_abs_grad):
    print(f"x{j+1}: {val:.6g}")
print("Most influential variable on average:", f"x{np.argmax(avg_abs_grad)+1}")


,x1,x2,x3,y,grad_x1,grad_x2,grad_x3,grad_norm,dominant_variable
13,0.600097,0.725136,0.066089,-0.036378,-0.368885,-0.774496,-0.611566,1.053534,x2
1,0.242114,0.644074,0.272433,-0.087963,0.449317,-0.312865,-0.395610,0.675483,x1
7,0.645503,0.397143,0.919771,-0.113869,0.501865,0.342861,-0.252345,0.658103,x1
3,0.492581,0.611593,0.340176,-0.034835,-0.179354,-0.572994,-0.246493,0.649037,x2
17,0.500000,0.500000,0.500000,-0.015979,0.187673,-0.505449,-0.358647,0.647555,x2
0,0.171525,0.343917,0.248737,-0.112122,0.342153,-0.158713,0.483900,0.613529,x3
15,0.754364,0.233177,0.303208,-0.092008,-0.304702,-0.215131,0.470489,0.600404,x3
12,0.046809,0.231360,0.770618,-0.118048,-0.015769,-0.448235,-0.365241,0.578415,x2
11,0.666014,0.671985,0.246295,-0.105965,-0.344510,-0.402058,-0.170363,0.556203,x2
10,0.220549,0.297825,0.343555,-0.046947,0.370004,-0.179730,0.304074,0.511533,x1


Average absolute gradient:
x1: 0.241486
x2: 0.268304
x3: 0.322402
Most influential variable on average: x3


In [7]:
# Use the neural network to propose a point by searching random candidates.
# For minimisation, choose low predicted target. If target is log_abs_y, this finds small magnitude, not necessarily negative y.
rng = np.random.default_rng(2)
candidates = rng.random((30000 if d <= 4 else 60000, d)).astype(np.float32)
cand_scaled = x_scaler.transform(candidates).astype(np.float32)
with torch.no_grad():
    pred_scaled = model(torch.tensor(cand_scaled)).numpy().ravel()
    pred_target = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()

nn_results = pd.DataFrame(candidates, columns=[f"x{i+1}" for i in range(d)])
nn_results[f"pred_{target_name}"] = pred_target
nn_results = nn_results.sort_values(f"pred_{target_name}", ascending=True)
display(nn_results.head(10))

best_nn = nn_results.iloc[0][[f"x{i+1}" for i in range(d)]].to_numpy(float)
print("NN suggested point:", np.round(best_nn, 6))
print("Portal format:", ", ".join(f"x{i+1}={v:.6f}" for i, v in enumerate(best_nn)))


,x1,x2,x3,pred_y
29678,0.136045,0.453321,0.991245,-0.399100
20452,0.104446,0.441574,0.997773,-0.398767
13314,0.087600,0.486871,0.998599,-0.398630
23929,0.123702,0.477793,0.993127,-0.398411
10035,0.149806,0.451400,0.986492,-0.398027
25392,0.160970,0.463161,0.987717,-0.397314
15322,0.065222,0.491689,0.989494,-0.395645
1734,0.099158,0.441514,0.986965,-0.395446
4250,0.065903,0.481203,0.985316,-0.394928
10780,0.194551,0.463132,0.990536,-0.394606


NN suggested point: [0.136045 0.453321 0.991245]
Portal format: x1=0.136045, x2=0.453321, x3=0.991245
